# 🚀 CinemaBox 70B/120B AI Server
### Powered by Google Storage (5 TB Drive) + Google Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/princesharmaofficial1-hub/cinemabox-colab/blob/main/deploy_70b_google_storage.ipynb)

This notebook turns your **5 TB Google Storage** and a **Colab GPU** into an always-accessible **OpenAI-compatible LLM Server** (`/v1/chat/completions`).

### Ready for:
1. **CinemaBox Flutter App** (Movie recaps, taste matching, plot analysis)
2. **VS Code (VSC)** (via Continue.dev, Roo Code, Cline, or Copilot custom endpoint)
3. **Any OpenAI client** (`curl`, Python, Web UI, etc.)

### Step 1: Verify Colab GPU & Mount Your 5 TB Google Drive
Click the play button below and tap **Connect to Google Drive** when prompted.

In [ ]:
import os
import subprocess
from google.colab import drive

print("🔍 Checking GPU hardware acceleration...")
gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"🎮 Detected GPU: {gpu_info}")

print("\n📁 Mounting your 5 TB Google Drive...")
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/CinemaBox_Models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print(f"\n✅ 5 TB Google Drive mounted successfully!")
print(f"📦 Models storage directory: {DRIVE_MODEL_DIR}")

### Step 2: Install High-Performance GPU Engine & Cloudflare Tunnel
Installs pre-compiled GPU inference packages and Cloudflare Tunnel for an instant, secure public HTTPS URL.

In [ ]:
!echo "⚡ Installing high-speed GPU engine and tools..."
# Install prebuilt CUDA wheels for ultra-fast startup (no slow compiler needed!)
!pip install --upgrade pip -q
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q
!pip install "llama-cpp-python[server]" airllm fastapi uvicorn huggingface_hub requests -q

# Download Cloudflare Tunnel (zero config, free public HTTPS)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ GPU Server & Cloudflare Tunnel installed successfully!")

### Step 3: Choose & Download Model to your 5 TB Google Drive
The model is downloaded directly into your **5 TB Google Drive**. Once downloaded, subsequent runs load **instantly** without re-downloading!

In [ ]:
from huggingface_hub import hf_hub_download
import os

# Select model configuration:
# Option A: LLaMA 3.3 70B Instruct (Ultimate powerhouse 70B parameter model)
REPO_ID = "bartowski/Llama-3.3-70B-Instruct-GGUF"
FILENAME = "Llama-3.3-70B-Instruct-Q4_K_M.gguf"

# Option B: Qwen 2.5 32B Instruct (Extremely fast, near-70B intelligence, fits completely in VRAM)
# REPO_ID = "bartowski/Qwen2.5-32B-Instruct-GGUF"
# FILENAME = "Qwen2.5-32B-Instruct-Q4_K_M.gguf"

model_path = os.path.join(DRIVE_MODEL_DIR, FILENAME)

if os.path.exists(model_path) and os.path.getsize(model_path) > 1024*1024*100:
    print(f"🎯 Model already exists in your 5 TB Google Drive at:\n   {model_path}")
    print(f"   Size: {os.path.getsize(model_path) / (1024**3):.2f} GB. Reusing stored weights!")
else:
    print(f"📥 Downloading {FILENAME} directly to your 5 TB Google Drive...")
    print("   (This takes a few minutes on Google's high-speed fiber backbone)")
    model_path = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILENAME,
        local_dir=DRIVE_MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print(f"\n✅ Download complete! Saved permanently in: {model_path}")

### Step 4: Start OpenAI-Compatible Server & Cloudflare HTTPS Tunnel
Starts the inference engine on port 8000 and exposes a global HTTPS URL.

In [ ]:
import subprocess
import time
import re
import os

# Launch OpenAI-compatible API server
server_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", model_path,
    "--n_gpu_layers", "35",          # Offload layers to Colab GPU
    "--n_ctx", "4096",                # Context window
    "--host", "127.0.0.1",
    "--port", "8000",
    "--api_key", "sk-cinemabox70b"
]

print("🚀 Launching OpenAI-compatible GPU server...")
server_proc = subprocess.Popen(server_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)  # Allow CUDA context initialization

# Start Cloudflare tunnel
print("🌐 Creating Cloudflare public HTTPS tunnel...")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cf_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*70)
    print("🎉 YOUR 70B AI SERVER IS LIVE AND ACCESSIBLE GLOBALLY!")
    print("="*70)
    print(f"\n👉 Base URL:           {public_url}/v1")
    print(f"👉 CinemaBox Endpoint: {public_url}/v1/chat/completions")
    print(f"👉 API Key:            sk-cinemabox70b")
    print(f"👉 Model ID:           Llama-3.3-70B-Instruct")
    print("\n" + "-"*70)
    print("📱 HOW TO USE IN CINEMABOX APP:")
    print(f"   1. Open CinemaBox on phone > Settings > On-Device Intelligence")
    print(f"   2. Switch to 'AirLLM 70B (Cloud)' tab")
    print(f"   3. Paste '{public_url}/v1/chat/completions' into Host API Endpoint")
    print(f"   4. Paste 'sk-cinemabox70b' into API Key")
    print(f"   5. Tap 'Test Connection & Generate' -> You are all set!")
    print("\n💻 HOW TO USE IN VS CODE (Roo Code / Continue / Cline):")
    print(f"   Provider: OpenAI Compatible")
    print(f"   Base URL: {public_url}/v1")
    print(f"   API Key:  sk-cinemabox70b")
    print(f"   Model:    Llama-3.3-70B-Instruct")
    print("="*70 + "\n")
else:
    print("❌ Cloudflare tunnel failed to produce a public URL. Retrying...")

### Step 5: Test Connection with Sample Movie Recommendation Query

In [ ]:
import requests

prompt = "Recommend 3 top sci-fi thriller films for a user who loves Interstellar and Inception. Include a 1-sentence hook for each."

print(f"💬 Sending test prompt: '{prompt}'\n")
res = requests.post(
    f"{public_url}/v1/chat/completions",
    headers={"Authorization": "Bearer sk-cinemabox70b"},
    json={
        "model": "Llama-3.3-70B-Instruct",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 300,
        "temperature": 0.7
    }
)

if res.status_code == 200:
    print("🎬 Model Output:\n")
    print(res.json()["choices"][0]["message"]["content"])
else:
    print(f"Error {res.status_code}: {res.text}")